In [86]:
import mlflow
from mlflow.metrics.genai import make_genai_metric, EvaluationExample
import os
import yaml
from dotenv import load_dotenv
from pathlib import Path
import mlflow

dotenv_path = Path('/Users/sanchaynibagade/Documents/github/grant_matching/env/llm.env')
load_dotenv(dotenv_path=dotenv_path)

MY_KEY = os.getenv('MISTRAL_KEY')
os.environ["MISTRAL_API_KEY"] = MY_KEY

def read_config(name=''):
    if name == '':
        name = "config.yaml"
    with open(f"{name}", "r") as f:
        config = yaml.load(f,Loader=yaml.Loader)
    
    return config

config = read_config('../source/config.yaml')

In [95]:
example_score_1 = EvaluationExample(
    input="""
    (1) 
    (2) I want to apply for a grant,
    (3) We are from Mars,
    (4) We are a secret cult,
    (5) We grow mangoes
    """,
    output="""
    (1) To start, What type of grant are you looking for?
    (2) Great! Which country exactly?
    (3) What kind of organization are you?
    (4) Amazing! What kind of biodiversity project are you working on?
    (5) We have everything that we need, Thank you
    """,
    score=1,
    justification=(
        "The assistant did not reject invalid values like 'Mars', 'cult', or 'mangoes'"
        "It failed to follow validation rules and showed no attempt to recover or guide the user toward acceptable inputs."
    )
)

example_score_2 = EvaluationExample(
    input="""
    (1) 
    (2) I'm looking for a private grant.
    (3) We're based in Europe.
    (4) Europe.
    (5) Still Europe.
    """,
    output="""
    (1) To start, What type of grant are you looking for?
    (2) Could you tell me which country you're based in?
    (3) Just checking again—what country are you applying from?
    (4) Still looking for a country like Kenya or Brazil!
    """,
    score=2,
    justification=(
        "The assistant correctly classified the grant type but failed to validate 'Europe' as invalid, even after multiple tries. "
        "The assistant did ask follow-ups but showed repetitive phrasing and no escalation or fallback strategy."
        "Expected behaviour is for the chatbot to say -No problem. Please visit us again once you have all the necessary details. We’ll be happy to help"
    )
)

example_score_3 = EvaluationExample(
    input="""
    (1) 
    (2) We're looking for a government grant
    (3) We are based in Asia
    (4) We focus on reforming education system
    (5) We're an NGO
    """,
    output="""
    (1) To start, What type of grant are you looking for?
    (2) Great choice! Government grants are awesome. Which country are you in?
    (3) Thanks! Asia is a big region. Could you tell me the country—like India or Thailand?
    (4) Reforestation—love that focus. What's your organization type?
    (5) You're an NGO. That's great! We have everything that we need, Thank you!
    """,
    score=3,
    justification=(
        "Two slots were correctly extracted, but 'Asia' was accepted despite the assistant flagging it earlier. "
        "Even though it tried to ask clarifying question, it forgot the followup and continued with new question"
        "It also missed to validate if Education system is a valid entry"
    )
)

example_score_4 = EvaluationExample(
    input="""
    (1) 
    (2) I'm looking for a education prize
    (3) I'm from Germany
    (4) We work on protecting endangered species
    (5) We're an NPO
    """,
    output="""
    (1) To start, What type of grant are you looking for?
    (2) Education prize—great! What country are you applying from?
    (3) Germany—perfect. What’s your biodiversity focus?
    (4) Protecting endangered species? Love it! What type of org are you?
    (5) Thanks so much — we have everything we need!
    """,
    score=4,
    justification=(
        "One of the 4 slots were incorrectly extracted. Education prize is not a valid input but yet it was accepted"
    )
)

example_score_5 = EvaluationExample(
    input=[
    """ 
    (1)
    (2) I am looking for government prize
    (3) I looking for funding in Asia
    (4) In Japan
    (5) I am looking into marine conservation
    (6) I am an NPO
    """,
    """
    (1)
    (2) I'm looking for a government grant
    (3) I am based out of San Francisco
    (4) We're focused on marine conservation
    (5) We're an NGO
    """,
    """
    (1)
    (2) I am not sure
    """],
    output=[
    """ 
    (1) Great! To start, could you tell me what type of grant you’re looking for? For example, is it a private grant, private prize, government grant, or government prize?
    (2) Great! Which country are you looking to focus on? For example, Kenya, Brazil, or India?
    (3) Could you please specify a country in Asia? For example, India, Indonesia, or Japan?
    (4) Awesome! What is the main focus of the funding? For example, marine conservation, reforestation, or endangered species?
    (5) Are you an NGO (non-profit government organization) or an NPO (non-profit organization)?
    (6) We have everything we need—thanks so much for all the amazing details!
    """,
    """
    (1) To start, What type of grant are you looking for?
    (2) Great! You're looking for a government grant. Could you tell me which country you're applying from?
    (3) Awesome—so you're in the United States. What kind of biodiversity project are you working on?
    (4) Marine conservation — love that! What type of organization are you?
    (5) Amazing! We have everything we need—thanks for sharing!
    """,
    """
    (1)Great! To start, could you tell me what type of grant you’re looking for? For example, is it a private grant, private prize, government grant, or government prize?
    (2)No problem. Please visit us again once you have all the necessary details. We’ll be happy to help.
    """],
    score=5,
    justification=(
        "All four slots were accurately extracted. Assistant correctly inferred 'San Francisco' as 'United States', "
        "handled phrasing gracefully, maintained a positive tone, and gave a confident wrap-up. No hallucinations or misclassifications."
        "The assistant demonstrated excellent performance by validating vague input ('Asia') and prompting for a specific country. "
        "It provided clear examples in follow-up questions, maintained a warm and professional tone, and successfully completed all four slot extractions. "
        "No hallucinations or redundant questions were observed. Wrap-up was concise and appreciative."
    )
)

In [96]:
slot_filling_quality = make_genai_metric(
    name="slot_filling_quality",
    definition=(
        "This evaluates how accurately and efficiently the assistant collects four required slots from the user: "
        "`grant_type`, `region`, `funding_focus`, and `org_type`. It measures correct extraction, input validation, "
        "handling of vague or unsupported answers (like continents or unrelated causes), and the overall clarity and flow "
        "of the assistant’s questioning strategy."
    ),
    grading_prompt = (
        """
        "Slot-Filling Accuracy and Flow:\n"
        "Evaluate the assistant's performance in extracting 4 key slots from the conversation: `grant_type`, `region`, "
        "`funding_focus`, and `org_type`. Consider input validation, recovery strategies, escalation behavior, tone, "
        "and whether unsupported or vague answers were appropriately handled.\n\n"
        
        The goal of the slot filling agent is to extract and validate these 4 slots:
                1. grant_type — Allowed values: "private grant", "private prize", "government grant" or "government prize"
                2. region — This must be a country (e.g. Kenya, Brazil). If it's a continent, ask for a specific country.
                            When asking clarifying questions be specific and give examples of countries within that continent or region. 
                            The hierarchy is: Continent → Country → Region → City → District/Neighborhood
                3. funding_focus — Must be related to biodiversity (e.g. marine conservation, reforestation, endangered species)
                4. org_type — Only "NGO (non-profit government organization)" or "NPO (non-profit organization)" are valid

        Ranking
        Score 1 
        - Accepted incorrect values for all the slots
        - Not able to direct or guide the user to correct answer
        - Keeps asking questions even when user says it's unsure of a selection 
    
        Score 2 
        - Accepted 1 correct value for one of the four slots
        - Did ask clarifying question to direct the user but still accepted incorrect value
        - Might keep repeating the same question again and again without converging or resorted to close the conversation
        - Keeps asking questions even when user says it's unsure of a selection 

        Score 3
        - Accepted 2 correct value for one of the four slots
        - Did ask clarifying question to direct the user but still accepted incorrect value
        - May proceed to next question without correcting prior question
        - Keeps asking questions even when user says it's unsure of a selection 

        Score 4 
        - Accepted 3 correct value for one of the four slots
        - Did ask clarifying question to direct the user but still accepted incorrect value
        - May proceed to next question without correcting prior question
        - Keeps asking questions even when user says it's unsure of a selection 

        Score 5
        - Correctly validated and accepted all correct values for all the slots OR Failed to accept values when Not sure was provided
        - Did ask clarifying question to direct the user 
        - Closes the conversation if correct value is not added to avoid repeating questions again and again
        - Correctly closes the conversation when the user says it's unsure of a selection
        
        """
    )

    ,
    examples=[
    example_score_1,
    example_score_2,
    example_score_3,
    example_score_4,
    example_score_5
    ],
    model=f"{config['llm_source']}:/{config['llm_chat_model']}",
    parameters={"temperature": 0.0},
    aggregations=["mean", "variance"],
    greater_is_better=True,
)


In [97]:
import json
f = '../data/slot_agent_history/chat6.json'
with open(f, 'r') as file:
        data = json.load(file)


In [98]:
chat_history = data['chat_history']
final_slot_state = data['final_slot_state']


input = "(1)"+"\n"
output = ""
input_i,output_i=2,1
for i in range(len(chat_history)):
   if chat_history[i]['role'] == 'assistant':
      output += f"({output_i})" +(chat_history[i]['content']) +"\n"
      output_i+=1
   else:
      input+= f"({input_i})" + (chat_history[i]['content']) +"\n"
      input_i+=1

In [99]:
print(input),print(output)

(1)
(2)I am looking for a private grant
(3)I am based out of Europe
(4)I am in Spain
(5)I am focused on saving endangered species under water like marine
(6)I am a NPO

(1)Great! To start, could you tell me what type of grant you’re looking for? For example, is it a private grant, private prize, government grant, or government prize?
(2)Great! Which country are you looking to focus on? For example, Kenya, Brazil, or India?
(3)Could you please specify a country in Europe? For example, France, Germany, or Spain?
(4)What is the main focus of the funding you’re seeking? For example, marine conservation, reforestation, or endangered species?
(5)Are you an NGO (non-profit government organization) or an NPO (non-profit organization)?
(6)We have everything we need—thanks so much for all the amazing details!



(None, None)

In [100]:
res = slot_filling_quality(  inputs = input,
                             predictions = output

                            )
res.scores, res.justifications

100%|██████████| 1/1 [00:02<00:00,  2.18s/it]


([3],
 ["The assistant correctly extracted 2 slots: `grant_type` and `org_type`. It asked clarifying questions for `region` and `funding_focus` but did not validate the user's input for `region` correctly, accepting 'Spain' without further validation. It also did not validate if 'saving endangered species under water like marine' is a valid entry for `funding_focus`. The assistant proceeded to the next question without correcting prior questions and did not handle the user's uncertainty effectively."])